In [2]:
import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import torch

ROOT = Path(r"D:\PS07_CT2MRI")

# Add project folders to Python path
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from resunet import ResidualUNet
from normalise import normalise_ct

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

PATIENT_ID = "P2658"

PATIENT_DIR = ROOT / "dataset" / "volumes" / PATIENT_ID

CHECKPOINT_PATH = ROOT / "models" / "baseline_resunet_l1.pth"


# --------------------------------------------------
# 2. Device
# --------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# --------------------------------------------------
# 3. Load CT volume
# --------------------------------------------------

ct_img = nib.load(
    str(PATIENT_DIR / "ct.nii.gz")
)

ct = np.asarray(
    ct_img.dataobj,
    dtype=np.float32
)

ct = normalise_ct(ct)

print("CT shape:", ct.shape)
print("Normalized CT range:", ct.min(), ct.max())


# --------------------------------------------------
# 4. Load model
# --------------------------------------------------

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Checkpoint epoch:", checkpoint["epoch"])


# --------------------------------------------------
# 5. Prepare one central slice
# --------------------------------------------------

z = ct.shape[2] // 2

ct_input = np.stack([
    ct[:, :, z - 1],
    ct[:, :, z],
    ct[:, :, z + 1]
], axis=0)

input_tensor = torch.from_numpy(
    ct_input.copy()
).unsqueeze(0).to(DEVICE)


# --------------------------------------------------
# 6. Generate prediction
# --------------------------------------------------

with torch.no_grad():

    prediction = model(input_tensor)

prediction = prediction.squeeze().cpu().numpy()


# --------------------------------------------------
# 7. Verify output
# --------------------------------------------------

print("\n========== INFERENCE TEST ==========")

print("Input shape:", tuple(input_tensor.shape))
print("Prediction shape:", prediction.shape)

print("Prediction minimum:", prediction.min())
print("Prediction maximum:", prediction.max())

print("Prediction finite:",
      np.isfinite(prediction).all())

print("\nInference test completed.")


Device: cuda
CT shape: (192, 224, 192)
Normalized CT range: 0.0 1.0
Checkpoint epoch: 10

========== INFERENCE TEST ==========
Input shape: (1, 3, 192, 224)
Prediction shape: (192, 224)
Prediction minimum: 1.3791934e-15
Prediction maximum: 0.91809297
Prediction finite: True

Inference test completed.
